# 📡 API Informes Agrícolas → Databricks

Notebook para importar en **Databricks** (Python). Lee los 6 endpoints de `http://172.10.18.128:9258/api/v1/index.php`, maneja la autenticación por `api_key` y la paginación automática, y opcionalmente persiste los datos en tablas **Delta**.

| Endpoint | Descripción | Filas aprox. |
|---|---|---|
| `plano` | Plano de siembra por finca/bloque/tabla | 10.7 K |
| `variedades` | Catálogo de variedades | 470 |
| `colores` | Catálogo de colores | 51 |
| `curvas_rosas` | Curvas paramétricas por variedad (JSON) | 27 |
| `proyecciones` | Proyección de producción diaria | 599 K |
| `lonas` | Producción por lona (tallos/plantas) | 1.75 M |

---

### ⚠️ Requisitos
- El cluster debe poder alcanzar `172.10.18.128:9258` (misma red/VNet, o firewall abierto). Prueba con `curl` desde una terminal del nodo de control si tienes dudas.
- Librerías: `requests` (incluida en el Databricks Runtime), `pandas`, PySpark (ya incluidas).
- Token del API: usar **Databricks Secrets** (recomendado) o definirlo en la celda de configuración.

## 1. Configuración

In [0]:
API_BASE = "http://190.60.223.98:9258/api/v1/index.php"

# ── Token del API ──────────────────────────────────────────────────────────────
# Opción A (recomendada): Databricks Secrets
#   1) Crea el scope:
#        databricks secrets create-scope informes_api
#   2) Guarda el token:
#        databricks secrets put-secret informes_api api_key --string-value "<TU_TOKEN>"
try:
    API_KEY = dbutils.secrets.get(scope="informes_api", key="api_key")
    print("API_KEY leída desde secrets 'informes_api'.")
except Exception:
    # Opción B (solo para pruebas locales / no versiones este notebook con el token real):
    API_KEY = "9035232260ca3fc11faa82b6b32b20605fe55c242e6903ca2f7e0887c3e9be17"
    print("No se encontró el secreto 'informes_api.api_key'. Usa la constante de respaldo o configura secrets.")

if API_KEY == "" or API_KEY.startswith("REEMPLAZA"):
    raise RuntimeError(
        "Configura el token: o crea el secret 'informes_api/api_key' o edita API_KEY en esta celda."
    )

# ── Delta ──────────────────────────────────────────────────────────────────────
DELTA_SCHEMA   = "proyecciones.default"            # catálogo/esquema destino (Unity Catalog o metastore)
WRITE_TO_DELTA = True                 # cambia a True para persistir en tablas Delta

# ── Modo de ejecución ──────────────────────────────────────────────────────────
# 'test'  → prueba rápida con un límite de páginas por endpoint
# 'full'  → descarga completa de todos los endpoints
MODE = "full"  # 'test' | 'full'

# ── Guards ─────────────────────────────────────────────────────────────────────
MAX_PAGES_TEST = 2       # páginas a leer por endpoint en modo test
HTTP_TIMEOUT   = 120     # segundos por request
HTTP_RETRIES   = 3       # reintentos por request (con backoff)

print(f"API_BASE={API_BASE}")
print(f"MODE={MODE} | WRITE_TO_DELTA={WRITE_TO_DELTA} | DELTA_SCHEMA={DELTA_SCHEMA}")

No se encontró el secreto 'informes_api.api_key'. Usa la constante de respaldo o configura secrets.
API_BASE=http://190.60.223.98:9258/api/v1/index.php
MODE=full | WRITE_TO_DELTA=True | DELTA_SCHEMA=proyecciones.default


## 2. Cliente HTTP con paginación automática

La API responde `{"status", "data", "total", "page", "pageSize", "filters", "generated_at"}`.
Cada endpoint tiene un límite de `pageSize` (500–5000). Este cliente itera todas las páginas hasta completar `total`.

In [0]:
import time
import requests
import pandas as pd
from pyspark.sql import SparkSession

# 'spark' ya existe en Databricks, pero dejamos la referencia explícita por claridad.
spark = SparkSession.builder.getOrCreate()


def fetch_endpoint(endpoint, page_size=5000, filters=None, max_pages=None, retries=None, timeout=None):
    """
    Descarga todas las páginas de un endpoint y devuelve (rows, total, pages_leidas).
    
    Soporta múltiples valores en filtros con sintaxis especial:
    - {"flor": ["ROC", "ROS"]} → hace 2 requests y une los resultados
    - {"flor": "ROC"} → un solo request

    endpoint  : nombre del endpoint ('plano', 'variedades', 'colores',
               'curvas_rosas', 'proyecciones', 'lonas')
    page_size : filas por página (respetando el tope del servidor)
    filters   : dict opcional de filtros GET, ej. {"fecha_inicio": "2026-01-01"}
                Si un valor es una lista, hace múltiples requests y une resultados.
    max_pages : tope de páginas (para pruebas). None = todas.
    """
    if retries is None:
        retries = HTTP_RETRIES
    if timeout is None:
        timeout = HTTP_TIMEOUT
    
    # Detectar si hay filtros con múltiples valores (listas)
    multi_value_filters = {}
    single_value_filters = {}
    
    if filters:
        for key, value in filters.items():
            if isinstance(value, list) and len(value) > 1:
                multi_value_filters[key] = value
            else:
                # Si es lista de un solo elemento, extraer el valor
                single_value_filters[key] = value[0] if isinstance(value, list) else value
    
    # Si hay filtros multi-valor, hacer requests separados y unir
    if multi_value_filters:
        if len(multi_value_filters) > 1:
            raise ValueError(f"Solo se soporta un filtro multi-valor a la vez. Encontrados: {list(multi_value_filters.keys())}")
        
        # Tomar el único filtro multi-valor
        filter_key = list(multi_value_filters.keys())[0]
        filter_values = multi_value_filters[filter_key]
        
        print(f"  → Filtro '{filter_key}' con múltiples valores: {filter_values}")
        print(f"  → Se harán {len(filter_values)} peticiones separadas...")
        
        all_rows = []
        total_sum = 0
        pages_sum = 0
        
        for i, single_value in enumerate(filter_values, 1):
            # Crear filtros para esta iteración
            current_filters = single_value_filters.copy()
            current_filters[filter_key] = single_value
            
            print(f"    [{i}/{len(filter_values)}] Descargando {filter_key}={single_value}...")
            rows, total, pages = _fetch_single_filter(
                endpoint, page_size, current_filters, max_pages, retries, timeout
            )
            
            all_rows.extend(rows)
            total_sum += total
            pages_sum += pages
            print(f"        ✓ {len(rows):,} filas descargadas")
        
        print(f"  → Total combinado: {len(all_rows):,} filas de {len(filter_values)} peticiones")
        return all_rows, total_sum, pages_sum
    
    # Si no hay multi-valor, hacer request normal
    return _fetch_single_filter(endpoint, page_size, single_value_filters or None, max_pages, retries, timeout)


def _fetch_single_filter(endpoint, page_size, filters, max_pages, retries, timeout):
    """
    Función auxiliar que hace la descarga con un único conjunto de filtros.
    """
    params = {"endpoint": endpoint, "pageSize": page_size, "api_key": API_KEY}
    if filters:
        params.update(filters)

    rows, page = [], 1
    while True:
        params["page"] = page
        payload = None
        for attempt in range(retries):
            try:
                resp = requests.get(API_BASE, params=params, timeout=timeout)
                resp.raise_for_status()
                payload = resp.json()
                break
            except Exception as exc:
                if attempt == retries - 1:
                    raise RuntimeError(f"GET {endpoint} (page={page}) falló: {exc}") from exc
                time.sleep(2 ** attempt)

        if not payload or payload.get("status") != "ok":
            raise RuntimeError(f"La API respondió un error en {endpoint} (page={page}): {payload}")

        data = payload.get("data") or []
        rows.extend(data)
        total = payload.get("total", len(rows))

        if len(rows) >= total or not data:
            return rows, total, page
        if max_pages and page >= max_pages:
            print(f"  * {endpoint}: detenido por max_pages={max_pages} ({len(rows)}/{total} filas)")
            return rows, total, page
        page += 1


def to_spark_df(rows, name):
    """Convierte la lista de dicts a DataFrame de Spark infiriendo el esquema."""
    if not rows:
        print(f"  ! {name}: sin datos (0 filas). DataFrame vacío.")
        return spark.createDataFrame([], "id long")
    pdf = pd.DataFrame(rows).fillna(value=pd.NA).fillna("")
    return spark.createDataFrame(pdf).replace("", None).dropDuplicates()


## 3. Configuración de endpoints

Ajusta `FACTORES_POR_ENDPOINT` si quieres acotar volúmenes (ej. filtrar `lonas` por rango de fecha en modo `full`).

### ✨ Filtros con múltiples valores

La API **NO soporta** múltiples valores separados por comas en un solo parámetro (ej. `flor=ROC,ROS` devuelve 0 filas).

**Solución implementada:**  
El código hace **peticiones separadas** para cada valor y une los resultados automáticamente.

**Sintaxis:**
```python
# ✅ CORRECTO - Usar lista de valores
{"flor": ["ROC", "ROS"]}

# ❌ INCORRECTO - No usar string con comas
{"flor": "ROC,ROS"}  # Esto devuelve 0 filas
```

**Ejemplo:**
```python
rows, total, pages = fetch_endpoint(
    "arreglos",
    page_size=5000,
    filters={"flor": ["ROC", "ROS"]},  # ← Descarga ambos automáticamente
)
# Resultado: hace 2 peticiones (flor=ROC y flor=ROS) y une las filas
```

**Restricción:** Solo se permite un filtro multi-valor a la vez. Si necesitas múltiples dimensiones, hazlo en varias llamadas.

In [0]:
# page_size debe respetar el tope del servidor por endpoint.
# IMPORTANTE: Los filtros con múltiples valores deben usar sintaxis de lista: ["valor1", "valor2"]
# La API NO soporta "valor1,valor2" - se harán peticiones separadas automáticamente.

ENDPOINTS = {
    "arreglos":      {"page_size": 5000, "filters": {"flor": ["ROC", "ROS"]}},
    "plano":         {"page_size": 5000, "filters": {"producto": ["ROSAS COLORES", "ROSAS ROJAS"]}},
    "variedades":    {"page_size": 2000, "filters": {"flor": ["ROC", "ROS"]}},
    "colores":       {"page_size": 500,  "filters": None},
    "curvas_rosas":  {"page_size": 1000, "filters": None},
    "dates":         {"page_size": 1000, "filters": None},
    "proyecciones":  {"page_size": 5000, "filters": {"flor": ["ROC", "ROS"]}},
    # Nota: fecha_inicio ajustada a 2026-09-01 (fecha real en los datos)
    # Si quieres datos históricos, verifica primero qué rango de fechas existe en el servidor
    "lonas":         {"page_size": 5000, "filters": {"flor": ["ROC", "ROS"], "fecha_inicio": "2022-01-01"}},
}

# En modo test se lee una fracción de cada endpoint.
if MODE == "test":
    FECTH_MAX_PAGES = MAX_PAGES_TEST
else:
    FECTH_MAX_PAGES = None

print("Endpoints registrados:", ", ".join(ENDPOINTS.keys()))

Endpoints registrados: arreglos, plano, variedades, colores, curvas_rosas, dates, proyecciones, lonas


## 4. Descarga de datos a DataFrames

Correr todas las celdas de esta sección en modo `full` puede tardar varios minutos.
**Nota**: `lonas` tiene filtros de fecha (desde 2022-01-01) y flor (ROC,ROS) para reducir el volumen.
Ejecuta primero la celda de verificación para estimar el tiempo de descarga.

In [0]:
# Prueba rápida: verificar que los filtros de lonas funcionan correctamente
import requests
import json

print("🔍 Verificando filtros del endpoint 'lonas'...\n")

# Hacer una petición pequeña para verificar
params = {
    "endpoint": "dates",
    "pageSize": 10,
    "page": 1,
    "api_key": API_KEY
}

# Agregar los filtros configurados
lonas_config = ENDPOINTS["lonas"]
if lonas_config.get("filters"):
    params.update(lonas_config["filters"])

print(f"Parámetros de prueba: {params}\n")

try:
    resp = requests.get(API_BASE, params=params, timeout=30)
    resp.raise_for_status()
    payload = resp.json()
    
    print(f"✓ Status: {payload.get('status')}")
    print(f"✓ Total de filas en servidor (con filtros): {payload.get('total'):,}")
    print(f"✓ Filtros aplicados por el servidor: {payload.get('filters')}")
    
    if payload.get("data"):
        print(f"\n📋 Estructura de una fila de ejemplo:")
        first_row = payload["data"][0]
        print(json.dumps(first_row, indent=2, ensure_ascii=False))
        print(f"\n📊 Campos disponibles: {list(first_row.keys())}")
        
        # Verificar que el filtro de flor funciona
        flor_values = set(row.get("flor") for row in payload["data"] if "flor" in row)
        if flor_values:
            print(f"\n✓ Valores de 'flor' en la muestra: {flor_values}")
        else:
            print("\n⚠️ WARNING: Campo 'flor' no encontrado en los datos")
        
        # Verificar fechas
        if "fecha" in first_row:
            print(f"\n✓ Rango de fechas en la muestra:")
            fechas = [row.get("fecha") for row in payload["data"] if "fecha" in row]
            print(f"  - Primera: {min(fechas)}")
            print(f"  - Última: {max(fechas)}")
        else:
            print("\n⚠️ WARNING: Campo 'fecha' no encontrado - verificar nombre del campo de fecha")
            print(f"   Campos disponibles: {list(first_row.keys())}")
    
    print(f"\n✅ Filtros verificados. Estimado de páginas a descargar: {payload.get('total', 0) // lonas_config['page_size'] + 1}")
    
except Exception as e:
    print(f"❌ Error al verificar filtros: {e}")
    print(f"   Revisa que el API esté disponible y que los nombres de filtros sean correctos")

🔍 Verificando filtros del endpoint 'lonas'...

Parámetros de prueba: {'endpoint': 'dates', 'pageSize': 10, 'page': 1, 'api_key': '9035232260ca3fc11faa82b6b32b20605fe55c242e6903ca2f7e0887c3e9be17', 'flor': ['ROC', 'ROS'], 'fecha_inicio': '2022-01-01'}

✓ Status: ok
✓ Total de filas en servidor (con filtros): 4,383
✓ Filtros aplicados por el servidor: []

📋 Estructura de una fila de ejemplo:
{
  "date": "2018-01-01 00:00:00",
  "year": 2018,
  "month": 1,
  "week": 1,
  "year_week": 1801,
  "year_week_iso": 201801,
  "day_of_week": 1,
  "day_of_week_num": 1,
  "day_name": "Lunes",
  "monday": "2018-01-01 00:00:00",
  "holiday": null
}

📊 Campos disponibles: ['date', 'year', 'month', 'week', 'year_week', 'year_week_iso', 'day_of_week', 'day_of_week_num', 'day_name', 'monday', 'holiday']

⚠️ WARNING: Campo 'flor' no encontrado en los datos

⚠️ WARNING: Campo 'fecha' no encontrado - verificar nombre del campo de fecha
   Campos disponibles: ['date', 'year', 'month', 'week', 'year_week', 'ye

In [0]:
dataframes = {}   # endpoint → (DataFrame, total, páginas)
dimensions = ["variedades", "colores", "curvas_rosas","dates"]   # catálogos pequeños
facts      = ["plano", "proyecciones", "lonas", "arreglos"]          # tablas grandes

# Configuración de timeouts por endpoint (algunos endpoints grandes necesitan más tiempo)
ENDPOINT_TIMEOUTS = {
    "lonas": 300,  # 5 minutos para endpoint con ~1.75M filas
}

order = dimensions + facts
for ep in order:
    cfg = ENDPOINTS[ep]
    página_start = time.time()
    endpoint_timeout = ENDPOINT_TIMEOUTS.get(ep, HTTP_TIMEOUT)
    print(f"[{ep}] descargando (pageSize={cfg['page_size']}, timeout={endpoint_timeout}s) ...")
    
    rows, total, pages = fetch_endpoint(
        ep,
        page_size=cfg["page_size"],
        filters=cfg["filters"],
        max_pages=FECTH_MAX_PAGES,
        timeout=endpoint_timeout,
    )
    
    df = to_spark_df(rows, ep)
    dataframes[ep] = (df, total, pages)
    print(f"  ✓ {ep}: {df.count()} filas en DataFrame | total servidor={total} | páginas={pages} | {time.time()-página_start:.1f}s")
    display(df.limit(5))

[variedades] descargando (pageSize=2000, timeout=120s) ...
  → Filtro 'flor' con múltiples valores: ['ROC', 'ROS']
  → Se harán 2 peticiones separadas...
    [1/2] Descargando flor=ROC...
        ✓ 470 filas descargadas
    [2/2] Descargando flor=ROS...
        ✓ 470 filas descargadas
  → Total combinado: 940 filas de 2 peticiones
  ✓ variedades: 470 filas en DataFrame | total servidor=940 | páginas=2 | 23.3s


codigo,nombre,codflor,codgcol,activo
5704,09S352,VCL,020,1
5957,14 ST 114 CRETA,VCL,003,1
2402,15 ST 463 PINK,CLA,003,1
5912,17 M 636 GREEN,VCM,021,1
5913,17 M 638 ORANGE,VCM,010,1


[colores] descargando (pageSize=500, timeout=120s) ...
  ✓ colores: 51 filas en DataFrame | total servidor=51 | páginas=1 | 1.0s


codigo,nombre,orden
020,AMARILLO,F - AMARILLO
TIC,AMARILLO CON ESCARCHA,null
022,AZUL,null
025,BICOLOR,null
026,BICOLOR AMARILLO,null


[curvas_rosas] descargando (pageSize=1000, timeout=120s) ...
  ✓ curvas_rosas: 27 filas en DataFrame | total servidor=27 | páginas=1 | 1.5s


id,variedad,ciclo,curva,s1,s2,s3,activo,fecha_creacion,porcentaje_ciegos
1,BLUSH,13,"[0.04, 0.11, 0.18, 0.3, 0.22, 0.1, 0.05]","[0.5, 0.5, 0, 0]","[0.25, 0.25, 0.25, 0.25]","[0, 0, 0.5, 0.5]",1,2026-08-13 13:34:12,0
2,BRIGHTON,12,"[0.04, 0.11, 0.18, 0.3, 0.22, 0.1, 0.05]","[0.5, 0.5, 0, 0]","[0.25, 0.25, 0.25, 0.25]","[0, 0, 0.5, 0.5]",1,2026-08-13 13:34:12,0
3,COOL WATER,11,"[0.04, 0.11, 0.18, 0.3, 0.22, 0.1, 0.05]","[0.5, 0.5, 0, 0]","[0.25, 0.25, 0.25, 0.25]","[0, 0, 0.5, 0.5]",1,2026-08-13 13:34:12,0
24,CUMBIA,12,"[0.04, 0.11, 0.18, 0.3, 0.22, 0.1, 0.05]","[0.5, 0.5, 0, 0]","[0.25, 0.25, 0.25, 0.25]","[0, 0, 0.5, 0.5]",1,2026-08-13 13:34:12,0
4,DEEP PURPLE,11,"[0.04, 0.11, 0.18, 0.3, 0.22, 0.1, 0.05]","[0.5, 0.5, 0, 0]","[0.25, 0.25, 0.25, 0.25]","[0, 0, 0.5, 0.5]",1,2026-08-13 13:34:12,0


[dates] descargando (pageSize=1000, timeout=120s) ...


---------------------------------------------------------------------------
ArrowTypeError                            Traceback (most recent call last)
File /databricks/python/lib/python3.12/site-packages/pyspark/sql/pandas/serializers.py:512, in ArrowStreamPandasSerializer._create_array(self, series, arrow_type, spark_type, arrow_cast)
    511 try:
--> 512     return pa.Array.from_pandas(
    513         series, mask=mask, type=arrow_type, safe=self._safecheck
    514     )
    515 except pa.lib.ArrowInvalid:

File /databricks/python/lib/python3.12/site-packages/pyarrow/array.pxi:1259, in pyarrow.lib.Array.from_pandas()

File /databricks/python/lib/python3.12/site-packages/pyarrow/array.pxi:365, in pyarrow.lib.array()

File /databricks/python/lib/python3.12/site-packages/pyarrow/array.pxi:91, in pyarrow.lib._ndarray_to_array()

File /databricks/python/lib/python3.12/site-packages/pyarrow/error.pxi:92, in pyarrow.lib.check_status()

ArrowTypeError: Expected bytes, got a 'float' object


## 5. Persistencia en Delta

Con `WRITE_TO_DELTA = True` guarda cada endpoint en una tabla Delta. El modo `overwrite` reemplaza la tabla completa (útil para refresco integral); `append` acumula (útil para cargas incrementales por fecha).

Los campos tipo `curva`, `s1`, `s2`, `s3` llegan como **strings JSON** (ej. `[0.04, 0.11, ...]`). Si el proyecto los necesita como arreglos, se pueden `from_json` en una celda posterior.

In [0]:
if WRITE_TO_DELTA:
    WRITE_MODE = "overwrite"  # 'overwrite' | 'append' | 'merge'
    for ep, (df, total, pages) in dataframes.items():
        table = f"{DELTA_SCHEMA}.{ep}"
        df.write.format("delta").mode(WRITE_MODE).saveAsTable(table)
        print(f"  ✓ {table}: {df.count()} filas guardadas")
else:
    print("WRITE_TO_DELTA=False → no se guardó nada. Cambia la variable en la celda de configuración y vuelve a ejecutar.")

  ✓ proyecciones.default.variedades: 470 filas guardadas
  ✓ proyecciones.default.colores: 51 filas guardadas
  ✓ proyecciones.default.curvas_rosas: 27 filas guardadas
  ✓ proyecciones.default.plano: 2610 filas guardadas
  ✓ proyecciones.default.proyecciones: 21541 filas guardadas
  ✓ proyecciones.default.lonas: 62856 filas guardadas
  ✓ proyecciones.default.arreglos: 10104 filas guardadas


## 6. Lectura posterior (recomendado en el proyecto de ciencia de datos)

Ejemplos de cómo consultar los datos ya persistidos desde otros notebooks.

In [0]:
# Ejemplo: leer directamente de Delta (si ya persististe)
# longitudes = spark.sql(f"SELECT * FROM {DELTA_SCHEMA}.informes_plano")

# Ejemplo: muestra de los DataFrames en memoria sin persistir
for ep in ["variedades", "colores", "curvas_rosas"]:
    df, total, pages = dataframes[ep]
    print(f"--- {ep} ({df.count()} filas) ---")
    display(df)

--- variedades (470 filas) ---


codigo,nombre,codflor,codgcol,activo
5704,09S352,VCL,020,1
5957,14 ST 114 CRETA,VCL,003,1
2402,15 ST 463 PINK,CLA,003,1
5912,17 M 636 GREEN,VCM,021,1
5913,17 M 638 ORANGE,VCM,010,1
5911,18 M 530 TERRA,VCM,025,1
6570,19 M 546,VCM,BUR,1
6527,20 ST 230,VCL,003,1
5921,2016 MB 8,VCM,001,1
5923,2017 M 17,VCM,003,1


--- colores (51 filas) ---


codigo,nombre,orden
020,AMARILLO,F - AMARILLO
TIC,AMARILLO CON ESCARCHA,null
022,AZUL,null
025,BICOLOR,null
026,BICOLOR AMARILLO,null
BPR,BICOLOR PURPURA,null
BCC,BICOLOR ROJO,null
BRB,BICOLOR ROSADO BLANCO,null
001,BLANCO,A - BLANCO
BUR,BURGUNDY,K - BURGUNDY


--- curvas_rosas (27 filas) ---


id,variedad,ciclo,curva,s1,s2,s3,activo,fecha_creacion,porcentaje_ciegos
1,BLUSH,13,"[0.04, 0.11, 0.18, 0.3, 0.22, 0.1, 0.05]","[0.5, 0.5, 0, 0]","[0.25, 0.25, 0.25, 0.25]","[0, 0, 0.5, 0.5]",1,2026-08-13 13:34:12,0
2,BRIGHTON,12,"[0.04, 0.11, 0.18, 0.3, 0.22, 0.1, 0.05]","[0.5, 0.5, 0, 0]","[0.25, 0.25, 0.25, 0.25]","[0, 0, 0.5, 0.5]",1,2026-08-13 13:34:12,0
3,COOL WATER,11,"[0.04, 0.11, 0.18, 0.3, 0.22, 0.1, 0.05]","[0.5, 0.5, 0, 0]","[0.25, 0.25, 0.25, 0.25]","[0, 0, 0.5, 0.5]",1,2026-08-13 13:34:12,0
24,CUMBIA,12,"[0.04, 0.11, 0.18, 0.3, 0.22, 0.1, 0.05]","[0.5, 0.5, 0, 0]","[0.25, 0.25, 0.25, 0.25]","[0, 0, 0.5, 0.5]",1,2026-08-13 13:34:12,0
4,DEEP PURPLE,11,"[0.04, 0.11, 0.18, 0.3, 0.22, 0.1, 0.05]","[0.5, 0.5, 0, 0]","[0.25, 0.25, 0.25, 0.25]","[0, 0, 0.5, 0.5]",1,2026-08-13 13:34:12,0
5,DEKORA,11,"[0.04, 0.11, 0.18, 0.3, 0.22, 0.1, 0.05]","[0.5, 0.5, 0, 0]","[0.25, 0.25, 0.25, 0.25]","[0, 0, 0.5, 0.5]",1,2026-08-13 13:34:12,0
6,ENGAGEMENT,11,"[0.04, 0.11, 0.18, 0.3, 0.22, 0.1, 0.05]","[0.5, 0.5, 0, 0]","[0.25, 0.25, 0.25, 0.25]","[0, 0, 0.5, 0.5]",1,2026-08-13 13:34:12,0
7,FREEDOM,12,"[0.04, 0.11, 0.18, 0.3, 0.22, 0.1, 0.05]","[0.5, 0.5, 0, 0]","[0.25, 0.25, 0.25, 0.25]","[0, 0, 0.5, 0.5]",1,2026-08-13 13:34:12,0
8,HIGH AND MAGIC,12,"[0.04, 0.11, 0.18, 0.3, 0.22, 0.1, 0.05]","[0.5, 0.5, 0, 0]","[0.25, 0.25, 0.25, 0.25]","[0, 0, 0.5, 0.5]",1,2026-08-13 13:34:12,0
9,JESSIKA,11,"[0.04, 0.11, 0.18, 0.3, 0.22, 0.1, 0.05]","[0.5, 0.5, 0, 0]","[0.25, 0.25, 0.25, 0.25]","[0, 0, 0.5, 0.5]",1,2026-08-13 13:34:12,0


## 7. Explorar

In [0]:
%sql
select * from proyecciones.default.colores

codigo,nombre,orden
022,AZUL,null
025,BICOLOR,null
TIC,AMARILLO CON ESCARCHA,null
BPR,BICOLOR PURPURA,null
020,AMARILLO,F - AMARILLO
026,BICOLOR AMARILLO,null
BRB,BICOLOR ROSADO BLANCO,null
001,BLANCO,A - BLANCO
BGR,BURGUNDY Y/O HOT PINK,null
BUR,BURGUNDY,K - BURGUNDY
